In [34]:
import torch
import torch.nn as nn

dtype = torch.float16

# Hyperparameters
d_batch = 4
d_context = 128
d_embed_features = 256
d_head_count = 2
d_head_features = d_embed_features // d_head_count  # 128
d_vocab = 4069
n_generate_steps = 32
d_max_context = d_context + n_generate_steps  # 160

# Tensor Shapes
s_tokens = (d_batch, d_context)  # (4, 128)
s_vocab = (d_vocab, d_embed_features)  # (4069, 256)
s_batch = (d_batch, d_context, d_embed_features)  # (4, 128, 256)
s_weight = (d_embed_features, d_embed_features)  # (256, 256)
s_wue = (d_embed_features, d_vocab)  # (256, 4069)
s_qkv = (d_batch, d_context, d_head_count, d_head_features)  # (4, 128, 2, 128)
s_single_qkv = (d_batch, 1, d_head_count, d_head_features)  # (4, 1, 2, 128)
s_kv_cache = (d_batch, d_head_count, d_max_context, d_head_features)  # (4, 2, 160, 128)

# 0.0 Parameters
vocab = nn.Parameter(torch.randn(s_vocab, dtype=dtype))  # (4069, 256)
wue = nn.Parameter(torch.randn(s_wue, dtype=dtype))  # (256, 4069)
wq = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)
wk = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)
wv = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)
wo = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)

# ==========================================
# 1.0 PREFILL STAGE (Full Context Processing)
# ==========================================

# 1.1 Input Lookup
tokens = torch.randint(0, d_vocab, s_tokens, dtype=torch.long)  # (4, 128)
batch = vocab[tokens]  # (4, 128, 256)

# 1.2 Q, K, V Projections
q = torch.matmul(batch, wq).reshape(s_qkv).transpose(1, 2)  # (4, 2, 128, 128)
k = torch.matmul(batch, wk).reshape(s_qkv).transpose(1, 2)  # (4, 2, 128, 128)
v = torch.matmul(batch, wv).reshape(s_qkv).transpose(1, 2)  # (4, 2, 128, 128)

# 1.3 Pre-Allocate KV Cache & Write Prefill Keys/Values
k_cache = torch.zeros(s_kv_cache, dtype=dtype)  # (4, 2, 160, 128)
v_cache = torch.zeros(s_kv_cache, dtype=dtype)  # (4, 2, 160, 128)

k_cache[:, :, :d_context, :] = k  # (4, 2, 128, 128)
v_cache[:, :, :d_context, :] = v  # (4, 2, 128, 128)
curr_pos = d_context  # 128

# 1.4 Attention Scores & Output Projection
attn = torch.matmul(q, k.transpose(-2, -1)) / (d_head_features ** 0.5)  # (4, 2, 128, 128)
attn = torch.softmax(attn, dim=-1)  # (4, 2, 128, 128)

out = torch.matmul(attn, v).transpose(1, 2).reshape(s_batch)  # (4, 128, 256)
out = torch.matmul(out, wo)  # (4, 128, 256)

# 1.5 Unembedding & First Token Selection
logits = torch.matmul(out, wue)  # (4, 128, 4069)
next_token = torch.argmax(logits[:, -1, :], dim=-1)[:, None]  # (4, 1)

print("--- 1.0 Prefill Completed ---")
print("Allocated K Cache shape:", k_cache.shape)  # torch.Size([4, 2, 160, 128])
print("First Predicted Token shape:", next_token.shape)  # torch.Size([4, 1])

# ==========================================
# 2.0 GENERATION STAGE (In-Place Buffer Writes)
# ==========================================

print("\n--- 2.0 Generation Loop Started ---")
for step in range(n_generate_steps):
    # 2.1 Single Token Lookup
    curr_batch = vocab[next_token]  # (4, 1, 256)

    # 2.2 Single Token Q, K, V Projections
    curr_q = torch.matmul(curr_batch, wq).reshape(s_single_qkv).transpose(1, 2)  # (4, 2, 1, 128)
    curr_k = torch.matmul(curr_batch, wk).reshape(s_single_qkv).transpose(1, 2)  # (4, 2, 1, 128)
    curr_v = torch.matmul(curr_batch, wv).reshape(s_single_qkv).transpose(1, 2)  # (4, 2, 1, 128)

    # 2.3 Direct In-Place Write into Pre-Allocated Buffer
    k_cache[:, :, curr_pos:curr_pos + 1, :] = curr_k  # (4, 2, 1, 128)
    v_cache[:, :, curr_pos:curr_pos + 1, :] = curr_v  # (4, 2, 1, 128)
    curr_pos += 1  # 129, 130, ..., 160

    # 2.4 Slice Valid Buffer Keys/Values for Attention
    active_k_cache = k_cache[:, :, :curr_pos, :]  # (4, 2, curr_pos, 128)
    active_v_cache = v_cache[:, :, :curr_pos, :]  # (4, 2, curr_pos, 128)

    # 2.5 Cached Attention Scores & Output Projection
    curr_attn = torch.matmul(curr_q, active_k_cache.transpose(-2, -1)) / (d_head_features ** 0.5)  # (4, 2, 1, curr_pos)
    curr_attn = torch.softmax(curr_attn, dim=-1)  # (4, 2, 1, curr_pos)

    curr_out = torch.matmul(curr_attn, active_v_cache).transpose(1, 2).reshape(d_batch, 1, d_embed_features)  # (4, 1, 256)
    curr_out = torch.matmul(curr_out, wo)  # (4, 1, 256)

    # 2.6 Unembedding & Next Token Selection
    curr_logits = torch.matmul(curr_out, wue)  # (4, 1, 4069)
    next_token = torch.argmax(curr_logits[:, -1, :], dim=-1)[:, None]  # (4, 1)

    print(f"Step 2.{step + 1:02d} | Valid Cache Slice: {active_k_cache.shape} | Next Token: {next_token.shape}")

--- 1.0 Prefill Completed ---
Allocated K Cache shape: torch.Size([4, 2, 160, 128])
First Predicted Token shape: torch.Size([4, 1])

--- 2.0 Generation Loop Started ---
Step 2.01 | Valid Cache Slice: torch.Size([4, 2, 129, 128]) | Next Token: torch.Size([4, 1])
Step 2.02 | Valid Cache Slice: torch.Size([4, 2, 130, 128]) | Next Token: torch.Size([4, 1])
Step 2.03 | Valid Cache Slice: torch.Size([4, 2, 131, 128]) | Next Token: torch.Size([4, 1])
Step 2.04 | Valid Cache Slice: torch.Size([4, 2, 132, 128]) | Next Token: torch.Size([4, 1])
Step 2.05 | Valid Cache Slice: torch.Size([4, 2, 133, 128]) | Next Token: torch.Size([4, 1])
Step 2.06 | Valid Cache Slice: torch.Size([4, 2, 134, 128]) | Next Token: torch.Size([4, 1])
Step 2.07 | Valid Cache Slice: torch.Size([4, 2, 135, 128]) | Next Token: torch.Size([4, 1])
Step 2.08 | Valid Cache Slice: torch.Size([4, 2, 136, 128]) | Next Token: torch.Size([4, 1])
Step 2.09 | Valid Cache Slice: torch.Size([4, 2, 137, 128]) | Next Token: torch.Size([4